# Regresión Logística: Detección de Noticias Falsas con Machine Learning

### Enunciado y contexto del ejercicio

Se propone la construcción de un sistema de aprendizaje automático capaz de predecir si una noticia determinada es **falsa (fake)** o **verdadera (real)**. Para ello, se utilizará un conjunto de datos de noticias etiquetadas y el algoritmo de **Regresión Logística**.

En secciones anteriores del curso hemos visto la **Regresión Lineal**, un algoritmo que nos permite predecir valores numéricos continuos (por ejemplo, el coste económico de un incidente de seguridad). Sin embargo, en muchos problemas del mundo real lo que queremos es **clasificar**: decidir a qué categoría pertenece un dato. Por ejemplo:
- ¿Este correo es spam o legítimo?
- ¿Esta transacción es fraudulenta o normal?
- ¿Esta noticia es falsa o verdadera?

La **Regresión Logística** es uno de los algoritmos más utilizados para resolver este tipo de problemas. A pesar de su nombre, no es un algoritmo de regresión, sino de **clasificación**. Lo que hace internamente es calcular la **probabilidad** de que un dato pertenezca a una clase determinada (por ejemplo, la probabilidad de que una noticia sea falsa), y en función de esa probabilidad, asigna una etiqueta.

En este ejercicio vamos a enfrentarnos además a un reto muy habitual en el mundo real: nuestros datos son **texto**. Los algoritmos de Machine Learning trabajan con números, no con palabras. Por lo tanto, tendremos que aprender a **transformar texto en números** antes de poder entrenar nuestro modelo. Este proceso se llama **vectorización** y lo iremos explicando paso a paso a lo largo del ejercicio.

### Conjunto de datos

##### [Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)

Este conjunto de datos contiene noticias recopiladas de diversas fuentes y está formado por **dos archivos CSV**:

- `True.csv`: contiene noticias **verdaderas** (reales).
- `Fake.csv`: contiene noticias **falsas**.

Cada archivo tiene las siguientes columnas:

| Columna | Descripción |
|---------|-------------|
| `title` | Título de la noticia |
| `text` | Cuerpo/contenido completo de la noticia |
| `subject` | Categoría temática de la noticia |
| `date` | Fecha de publicación |

En total, el conjunto de datos contiene más de **44.000 noticias**, aproximadamente la mitad falsas y la mitad reales.

**Descarga**: Puedes descargar el conjunto de datos desde [Kaggle](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset). Una vez descargado, coloca los archivos `True.csv` y `Fake.csv` en el mismo directorio que este notebook.

### 1. Lectura del conjunto de datos

Antes de comenzar a realizar ninguna acción, debemos leer el conjunto de datos de disco y cargarlo en memoria para poder trabajar con él. A diferencia de otros conjuntos de datos que hemos visto en el curso, en este caso la información nos viene separada en dos archivos CSV: uno con las noticias verdaderas y otro con las falsas.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Implementa el código en Python necesario para leer los dos archivos CSV del conjunto de datos, unirlos en un único DataFrame y añadir una columna que indique si cada noticia es falsa o verdadera.
</div>

**Pista**: Utiliza la librería `pandas` para leer los archivos CSV con `pd.read_csv()`. Después, añade a cada DataFrame una columna llamada `label` con el valor `"REAL"` o `"FAKE"` según corresponda. Finalmente, utiliza `pd.concat()` para unir ambos DataFrames en uno solo.

In [1]:
# IMPORTANTE: Ejecuta esta celda para instalar las dependencias antes de comenzar
%pip install pandas beautifulsoup4 nltk scikit-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 14.1 MB/s  0:00:00

   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   ---------------------------------------- 0/5 [tqdm]
   -------- ------------------------------- 1/5 [soupsieve]
   ---------------- ----------------------- 2/5 [regex]
   ---------------- ----------------------- 2/5 [regex]
   ------------------------ --------------- 3/5 [nltk]
   ------------------------ --------------- 3/5 [nltk]
   ------------------------ --------------- 3/5 [nltk]
   ------------------------ --------------- 3/5 [nltk]
   ------------------------ -----

In [1]:
import pandas as pd

In [2]:
# Leemos los dos archivos CSV
df_true = pd.read_csv("datasets/Fake_Real_News_Dataset/True.csv")
df_fake = pd.read_csv("datasets/Fake_Real_News_Dataset/Fake.csv")

In [3]:
# Exploramos las primeras filas del conjunto de noticias verdaderas
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [4]:
# Exploramos las primeras filas del conjunto de noticias falsas
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
# Añadimos una columna "label" a cada DataFrame para identificar las noticias
df_true["label"] = "REAL"
df_fake["label"] = "FAKE"  

In [6]:
# Unimos ambos DataFrames en uno solo
df = pd.concat([df_true, df_fake], ignore_index=True)

In [7]:
# Verificamos el resultado
print(f"Noticias verdaderas: {len(df_true)}")
print(f"Noticias falsas: {len(df_fake)}")
print(f"Total de noticias: {len(df)}")

Noticias verdaderas: 21417
Noticias falsas: 23481
Total de noticias: 44898


In [8]:
df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",REAL
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",REAL
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",REAL
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",REAL
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",REAL


In [9]:
df.tail()

,title,text,subject,date,label
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",FAKE
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",FAKE
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",FAKE
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",FAKE
44897,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",FAKE


### 2. Exploración del conjunto de datos

Antes de aplicar cualquier técnica de Machine Learning, siempre es recomendable realizar una exploración inicial de los datos. Esto nos permite entender mejor con qué estamos trabajando y detectar posibles problemas antes de que afecten a nuestro modelo.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Explora el conjunto de datos: revisa cuántas noticias hay de cada tipo, comprueba si hay valores nulos y visualiza algunos ejemplos de noticias falsas y verdaderas.
</div>

In [10]:
# Distribución de noticias por etiqueta
df["label"].value_counts()

label
FAKE    23481
REAL    21417
Name: count, dtype: int64

In [11]:
# Veamos un ejemplo de noticia REAL
print("=" * 80)
print("NOTICIA REAL")
print("=" * 80)
print(f"\nTítulo: {df[df['label']=='REAL'].iloc[0]['title']}")
print(f"\nTexto: {df[df['label']=='REAL'].iloc[0]['text'][:500]}...")

NOTICIA REAL

Título: As U.S. budget fight looms, Republicans flip their fiscal script

Texto: WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they retur...


In [12]:
# Veamos un ejemplo de noticia FAKE
print("=" * 80)
print("NOTICIA FAKE")
print("=" * 80)
print(f"\nTítulo: {df[df['label']=='FAKE'].iloc[0]['title']}")
print(f"\nTexto: {df[df['label']=='FAKE'].iloc[0]['text'][:500]}...")

NOTICIA FAKE

Título:  Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing

Texto: Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a gr...


**Observación**: Si te fijas en las noticias falsas, muchas de ellas contienen **restos de código HTML**, URLs, caracteres especiales y otros elementos que no forman parte del texto real de la noticia. Esto es muy habitual cuando se recopilan datos de internet. En la siguiente sección vamos a aprender a **limpiar** este texto.

### 3. Procesamiento del texto

Aquí nos encontramos con uno de los **grandes retos** del Machine Learning aplicado a texto: los datos no vienen limpios. Las noticias pueden contener:

- **Código HTML** (etiquetas como `<p>`, `<br>`, `<div>`, etc.)
- **URLs** (enlaces a páginas web)
- **Signos de puntuación** que no aportan información útil
- **Palabras muy comunes** que aparecen en todas las noticias y no ayudan a distinguir entre falsas y verdaderas (por ejemplo: "the", "is", "and", "a"...)

Antes de poder entrenar nuestro modelo, necesitamos **procesar y limpiar el texto** para quedarnos únicamente con la información relevante. Vamos a hacerlo paso a paso.

#### 3.1. Eliminación de código HTML

Muchas noticias, especialmente las recopiladas de internet, pueden contener restos de código HTML. Por ejemplo, en lugar de ver un párrafo limpio, podemos encontrarnos algo como:

```
<p>This is a <b>breaking</b> news story...</p>
```

Necesitamos una forma de quedarnos solo con el texto y eliminar todas las etiquetas HTML.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Implementa una función en Python que elimine todas las etiquetas HTML de un texto dado.
</div>

**Pista**: Puedes utilizar la librería `BeautifulSoup` del paquete `bs4`, que nos permite parsear HTML fácilmente. La función `BeautifulSoup(texto, "html.parser").get_text()` devuelve únicamente el texto sin etiquetas.

In [15]:
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

In [16]:
# Ejemplo de texto con HTML
ejemplo_html = '<p>This is a <b>breaking</b> news <a href="http://example.com">story</a></p>'
print("Texto original:", ejemplo_html)

Texto original: <p>This is a <b>breaking</b> news <a href="http://example.com">story</a></p>


In [17]:
# Eliminamos las etiquetas HTML
texto_limpio = BeautifulSoup(ejemplo_html, "html.parser").get_text()
print("Texto limpio:", texto_limpio)

Texto limpio: This is a breaking news story


In [18]:
def strip_html(text):
    """Elimina las etiquetas HTML de un texto."""
    return BeautifulSoup(text, "html.parser").get_text()

In [20]:
# Probamos con una noticia real del conjunto de datos
print(strip_html(df.iloc[0]["text"])[:500])

WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they retur


#### 3.2. Eliminación de URLs

Las noticias también pueden contener URLs (direcciones web) que no aportan información útil para nuestro clasificador. Necesitamos eliminarlas.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Implementa una función en Python que elimine todas las URLs de un texto dado.
</div>

**Pista**: Puedes utilizar **expresiones regulares** con la librería `re`. El patrón `https?://\S+` captura la mayoría de URLs (busca texto que empiece por `http://` o `https://` seguido de cualquier carácter que no sea un espacio).

In [21]:
import re

In [22]:
# Ejemplo de texto con URL
ejemplo_url = "Breaking news about the economy https://www.example.com/news check it out"
print("Texto original:", ejemplo_url)

Texto original: Breaking news about the economy https://www.example.com/news check it out


In [23]:
# Eliminamos las URLs
texto_sin_urls = re.sub(r'https?://\S+', '', ejemplo_url)
print("Texto limpio:", texto_sin_urls)

Texto limpio: Breaking news about the economy  check it out


In [24]:
def remove_urls(text):
    """Elimina las URLs de un texto."""
    return re.sub(r'https?://\S+', '', text)

#### 3.3. Eliminación de signos de puntuación y conversión a minúsculas

Los signos de puntuación (comas, puntos, signos de exclamación, etc.) generalmente no aportan información útil para distinguir entre noticias falsas y verdaderas. Además, queremos que nuestro algoritmo trate igual las palabras independientemente de si están en mayúsculas o minúsculas (por ejemplo, "President" y "president" deberían ser la misma palabra).

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Implementa una función que convierta el texto a minúsculas y elimine los signos de puntuación.
</div>

**Pista**: Utiliza el método `.lower()` de Python para convertir a minúsculas y la librería `string` para obtener una lista de todos los signos de puntuación.

In [25]:
import string

# Estos son todos los signos de puntuación que Python reconoce
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [26]:
def remove_punctuation(text):
    """Convierte el texto a minúsculas y elimina los signos de puntuación."""
    text = text.lower()
    # Eliminamos la puntuación estándar ASCII
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Eliminamos también las comillas tipográficas y otros caracteres especiales Unicode
    # que no están incluidos en string.punctuation
    text = re.sub(r'[\u2018\u2019\u201C\u201D\u2013\u2014\u2026]', '', text)
    return text

In [29]:
print('[\u2018 \u2019 \u201C \u201D \u2013 \u2014 \u2026]')

[‘ ’ “ ” – — …]


In [30]:
# Ejemplo
ejemplo = "BREAKING NEWS! The President said: 'No comment'."
print("Original:", ejemplo)
print("Procesado:", remove_punctuation(ejemplo))

Original: BREAKING NEWS! The President said: 'No comment'.
Procesado: breaking news the president said no comment


#### 3.4. Eliminación de stopwords y Stemming

Ahora vamos a aplicar dos técnicas fundamentales del **Procesamiento de Lenguaje Natural (NLP)** que veremos con más profundidad en la siguiente sección del curso. De momento, vamos a introducirlas de manera práctica:

**Stopwords (palabras vacías)**: Son palabras muy comunes en un idioma que aparecen en prácticamente todos los textos y, por lo tanto, no aportan información útil para distinguir entre categorías. Algunos ejemplos en inglés son: *"the"*, *"is"*, *"and"*, *"a"*, *"in"*, *"to"*...

Imagina que estás intentando distinguir si una noticia es falsa o verdadera. La palabra *"the"* va a aparecer en ambos tipos de noticias con una frecuencia similar, así que no nos ayuda. Sin embargo, palabras como *"hoax"* (engaño) o *"verified"* (verificado) podrían ser más informativas.

**Stemming (lematización)**: Es el proceso de reducir una palabra a su **raíz**. Por ejemplo:
- *"running"*, *"runs"*, *"ran"* → **"run"**
- *"playing"*, *"played"*, *"plays"* → **"play"**

¿Por qué nos interesa esto? Porque para nuestro algoritmo, *"running"* y *"runs"* son palabras completamente diferentes, cuando en realidad tienen el mismo significado base. Al reducirlas a su raíz, agrupamos las variantes de una misma palabra.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Explora la librería <code>nltk</code> e implementa la eliminación de stopwords y el proceso de stemming.
</div>

**Pista**: Revisa la clase `PorterStemmer()` de nltk y el listado de stopwords en `nltk.corpus.stopwords.words('english')`. Para dividir un texto en palabras individuales, utiliza `nltk.tokenize.word_tokenize()`.

In [31]:
import nltk

# Descargamos los recursos necesarios de nltk (solo la primera vez)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [32]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [33]:
# Veamos algunas stopwords en inglés
stop_words = set(stopwords.words('english'))
print(f"Número de stopwords: {len(stop_words)}")
print(f"\nAlgunos ejemplos: {list(stop_words)[:20]}")

Número de stopwords: 198

Algunos ejemplos: ['the', 'had', "she'll", 'these', "we've", 'with', 'can', 'nor', "i've", "won't", 'hasn', 'aren', 'myself', 'as', 's', 't', "she's", 'again', 'being', 'doing']


In [34]:
# Veamos cómo funciona el Stemmer
stemmer = PorterStemmer()

palabras = ["running", "runs", "ran", "playing", "played", "investigation", "investigating"]
for palabra in palabras:
    print(f"  {palabra:20s} → {stemmer.stem(palabra)}")

  running              → run
  runs                 → run
  ran                  → ran
  playing              → play
  played               → play
  investigation        → investig
  investigating        → investig


In [35]:
# Ejemplo completo: tokenizar, eliminar stopwords y aplicar stemming
ejemplo = "The president was running an investigation into the reported claims"
print("Texto original:", ejemplo)

# 1. Tokenizar (dividir en palabras)
tokens = word_tokenize(ejemplo.lower())
print("\nTokens:", tokens)

# 2. Eliminar stopwords
tokens_filtrados = [t for t in tokens if t not in stop_words]
print("\nSin stopwords:", tokens_filtrados)

# 3. Aplicar stemming
tokens_stem = [stemmer.stem(t) for t in tokens_filtrados]
print("\nCon stemming:", tokens_stem)

Texto original: The president was running an investigation into the reported claims

Tokens: ['the', 'president', 'was', 'running', 'an', 'investigation', 'into', 'the', 'reported', 'claims']

Sin stopwords: ['president', 'running', 'investigation', 'reported', 'claims']

Con stemming: ['presid', 'run', 'investig', 'report', 'claim']


### 4. Función de preprocesamiento completa

Ahora que hemos implementado todas las transformaciones de texto por separado, vamos a ponerlas todas en común dentro de una única función que procese una noticia de principio a fin.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Implementa una función en Python que aplique todas las transformaciones anteriores sobre un texto: eliminación de HTML, eliminación de URLs, conversión a minúsculas, eliminación de puntuación, eliminación de stopwords y stemming. La función debe devolver el texto procesado como un único string.
</div>

In [42]:
def preprocesar_texto(text):
    """
    Aplica todas las transformaciones de preprocesamiento sobre un texto:
    0. Eliminar prefijos de fuentes
    1. Eliminar etiquetas HTML
    2. Eliminar URLs
    3. Convertir a minúsculas
    4. Eliminar signos de puntuación
    5. Tokenizar
    6. Eliminar stopwords
    7. Aplicar stemming
    
    Devuelve el texto procesado como un único string.
    """
    # 0. Eliminar prefijos de fuente tipo "CIUDAD (Reuters) -" o "CIUDAD (AP) -"
    text = re.sub(r'^[A-Z\s,.]+ \([^)]+\)\s*[-–—]?\s*', '', text)
    # 1. Eliminar HTML
    text = strip_html(text)
    # 2. Eliminar URLs
    text = remove_urls(text)
    # 3 y 4. Minúsculas y puntuación
    text = remove_punctuation(text)
    # 5. Tokenizar
    tokens = word_tokenize(text)
    # 6. Eliminar stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # 7. Stemming
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(t) for t in tokens]
    # Unimos los tokens de nuevo en un string
    return " ".join(tokens)

In [43]:
# Probamos con una noticia del conjunto de datos
texto_original = df.iloc[0]["text"]
print("TEXTO ORIGINAL (primeros 300 caracteres):")
print(texto_original[:300])
print("\n" + "=" * 80)
print("\nTEXTO PROCESADO (primeros 300 caracteres):")
print(preprocesar_texto(texto_original)[:300])

TEXTO ORIGINAL (primeros 300 caracteres):
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


TEXTO PROCESADO (primeros 300 caracteres):
head conserv republican faction us congress vote month huge expans nation debt pay tax cut call fiscal conserv sunday urg budget restraint 2018 keep sharp pivot way among republican us repres mark meadow speak cb face nation drew hard line feder spend lawmak brace battl januari return holiday wednes


Como puedes ver, el texto procesado es mucho más limpio y contiene únicamente las **palabras relevantes reducidas a su raíz**. Este es el texto que nuestro algoritmo de Machine Learning va a recibir como entrada.

### 5. Aplicar el preprocesamiento al conjunto de datos

Ahora que tenemos nuestra función de preprocesamiento, vamos a aplicarla sobre todas las noticias del conjunto de datos. Este proceso puede tardar un poco ya que tenemos más de 44.000 noticias.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Aplica la función de preprocesamiento a todas las noticias del conjunto de datos. <b>Importante:</b> Para no tener que esperar demasiado tiempo, comienza trabajando con un subconjunto reducido de datos (por ejemplo, 1000 noticias).
</div>

**Pista**: Utiliza el método `.apply()` de Pandas para aplicar la función de preprocesamiento sobre la columna `"text"` del DataFrame. Para empezar, trabaja con un subconjunto usando `.sample()` o seleccionando las primeras N filas.

In [44]:
# Para empezar, trabajamos con un subconjunto de 1000 noticias
# Mezclamos el DataFrame para tener noticias falsas y verdaderas mezcladas
df_sample = df.sample(n=1000, random_state=42)

print(f"Tamaño del subconjunto: {len(df_sample)}")
print(f"\nDistribución de etiquetas:")
print(df_sample["label"].value_counts())

Tamaño del subconjunto: 1000

Distribución de etiquetas:
label
FAKE    536
REAL    464
Name: count, dtype: int64


In [45]:
# Aplicamos el preprocesamiento a todas las noticias del subconjunto
# Esto puede tardar unos segundos
print("Preprocesando noticias...")
df_sample["text_clean"] = df_sample["text"].apply(preprocesar_texto)
print("¡Listo!")

Preprocesando noticias...
¡Listo!


In [46]:
# Veamos el resultado
df_sample[["text", "text_clean", "label"]].head()

,text,text_clean,label
22216,"Donald Trump s White House is in chaos, and th...",donald trump white hous chao tri cover russia ...,FAKE
27917,Now that Donald Trump is the presumptive GOP n...,donald trump presumpt gop nomine time rememb c...,FAKE
25007,Mike Pence is a huge homophobe. He supports ex...,mike penc huge homophob support exgay convers ...,FAKE
1377,SAN FRANCISCO (Reuters) - California Attorney ...,california attorney gener xavier becerra said ...,REAL
32476,Twisted reasoning is all that comes from Pelos...,twist reason come pelosi day especi 2006 promi...,FAKE


In [47]:
# Ejemplo de antes y después
print("ANTES:")
print(df_sample.iloc[0]["text"][:200])
print("\nDESPUÉS:")
print(df_sample.iloc[0]["text_clean"][:200])

ANTES:
Donald Trump s White House is in chaos, and they are trying to cover it up. Their Russia problems are mounting by the hour, and they refuse to acknowledge that there are problems surrounding all of th

DESPUÉS:
donald trump white hous chao tri cover russia problem mount hour refus acknowledg problem surround fake news hoax howev fact bear thing differ seem crack congression public leadershipchuck grassley ri


### 6. Codificando el texto: vectorización

Este es uno de los pasos más **importantes** y que puede resultar más novedoso. Hasta ahora, los conjuntos de datos con los que hemos trabajado en el curso tenían columnas numéricas (precio, número de equipos, etc.). Sin embargo, en este caso nuestros datos son **texto** y los algoritmos de Machine Learning necesitan recibir **números**.

¿Cómo podemos convertir texto en números? A este proceso se le llama **vectorización** y existen varias técnicas. Nosotros vamos a utilizar una de las más sencillas e intuitivas: **Bag of Words (Bolsa de Palabras)**.

#### ¿Qué es Bag of Words?

La idea es muy sencilla. Imagina que tenemos solo dos noticias en nuestro conjunto de datos:

- **Noticia 1**: *"president signs new law today"*
- **Noticia 2**: *"new study shows results today"*

Lo que hace Bag of Words es:

1. **Crear un vocabulario** con todas las palabras únicas que aparecen en el conjunto de datos: `["law", "new", "president", "results", "shows", "signs", "study", "today"]`

2. **Representar cada noticia como un vector** de números, donde cada número indica **cuántas veces aparece cada palabra** del vocabulario en esa noticia:

| | law | new | president | results | shows | signs | study | today |
|---|---|---|---|---|---|---|---|---|
| **Noticia 1** | 1 | 1 | 1 | 0 | 0 | 1 | 0 | 1 |
| **Noticia 2** | 0 | 1 | 0 | 1 | 1 | 0 | 1 | 1 |

De esta manera, cada noticia se convierte en un **vector numérico** que el algoritmo de Machine Learning puede entender. A este proceso se le llama **vectorización** y es fundamental cuando trabajamos con texto.

En la siguiente sección del curso profundizaremos mucho más en estas técnicas. De momento, vamos a utilizar la implementación que nos proporciona Sklearn.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Aplica la técnica de vectorización Bag of Words sobre los textos procesados para transformarlos en una representación numérica que pueda ser utilizada por el algoritmo de Machine Learning.
</div>

**Pista**: Revisa la clase `CountVectorizer` de Sklearn. Esta clase implementa exactamente el proceso de Bag of Words que hemos descrito. Tiene dos métodos principales:
- `.fit()`: aprende el vocabulario a partir de los textos de entrenamiento.
- `.transform()`: transforma los textos en vectores numéricos usando el vocabulario aprendido.

In [49]:
from sklearn.feature_extraction.text import CountVectorizer

In [51]:
# Primero, veamos un ejemplo sencillo para entender cómo funciona CountVectorizer
ejemplo_textos = [
    "president signs new law today",
    "new study shows results today"
]

vectorizer_ejemplo = CountVectorizer()
vectorizer_ejemplo.fit(ejemplo_textos)

print("Vocabulario aprendido:")
print(vectorizer_ejemplo.get_feature_names_out())

Vocabulario aprendido:
['law' 'new' 'president' 'results' 'shows' 'signs' 'study' 'today']


In [52]:
# Transformamos los textos en vectores
vectores_ejemplo = vectorizer_ejemplo.transform(ejemplo_textos)

# Visualizamos la matriz resultante
pd.DataFrame(
    vectores_ejemplo.toarray(), 
    columns=vectorizer_ejemplo.get_feature_names_out(),
    index=["Noticia 1", "Noticia 2"]
)

,law,new,president,results,shows,signs,study,today
Noticia 1,1,1,1,0,0,1,0,1
Noticia 2,0,1,0,1,1,0,1,1


Como puedes ver, el resultado es exactamente la tabla que hemos descrito antes. Cada fila representa una noticia y cada columna representa una palabra del vocabulario. Los valores indican cuántas veces aparece cada palabra en cada noticia.

Ahora vamos a aplicar este mismo proceso sobre nuestro conjunto de datos real.

In [53]:
# Aplicamos CountVectorizer sobre nuestros textos procesados
vectorizer = CountVectorizer()
vectorizer.fit(df_sample["text_clean"])

print(f"Tamaño del vocabulario: {len(vectorizer.get_feature_names_out())} palabras únicas")

Tamaño del vocabulario: 19184 palabras únicas


In [54]:
# Transformamos los textos en vectores
X_vect = vectorizer.transform(df_sample["text_clean"])

print(f"Dimensiones de la matriz resultante: {X_vect.shape}")
print(f"  → {X_vect.shape[0]} noticias")
print(f"  → {X_vect.shape[1]} palabras en el vocabulario")

Dimensiones de la matriz resultante: (1000, 19184)
  → 1000 noticias
  → 19184 palabras en el vocabulario


In [55]:
# Veamos las primeras filas de la matriz (solo las primeras 10 columnas)
pd.DataFrame(
    X_vect.toarray(), 
    columns=vectorizer.get_feature_names_out()
)

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Nota**: La matriz resultante es muy grande (cada noticia se representa con un vector de miles de elementos) y la mayoría de los valores son 0 (una noticia individual solo contiene una pequeña fracción de todas las palabras del vocabulario). Este tipo de matrices se llaman **matrices dispersas (sparse)** y Sklearn las maneja de forma eficiente internamente.

### 7. Entrenamiento del algoritmo

¡Enhorabuena! Ya tienes la parte más complicada del ejercicio realizada. Hemos conseguido transformar las noticias de texto en vectores numéricos que nuestro algoritmo de Machine Learning puede procesar.

Ahora vamos a entrenar un modelo de **Regresión Logística** que aprenderá de estos vectores a clasificar las noticias en falsas y verdaderas. El proceso es exactamente igual que el que hemos visto con la Regresión Lineal: le proporcionamos datos de entrenamiento (noticias con sus etiquetas) y el algoritmo aprende los patrones que distinguen una noticia falsa de una verdadera.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Utiliza el algoritmo de Machine Learning <code>LogisticRegression</code> para clasificar entre noticias falsas y verdaderas. Puedes encontrar la implementación de este algoritmo en Sklearn.
</div>

**Pista 1**: Comienza utilizando un número reducido de noticias para que no tarde demasiado tiempo, por ejemplo 1000 noticias. Aplica todas las transformaciones que hemos implementado anteriormente sobre ellas.

In [56]:
# Leemos únicamente un subconjunto de 1500 noticias
df_all = df.sample(n=1500, random_state=42)

# Nos quedamos con 1000 noticias para el entrenamiento
df_sample = df_all.iloc[:1000]

print(f"Tamaño del subconjunto: {len(df_sample)}")
print(f"\nDistribución de etiquetas:")
print(df_sample["label"].value_counts())

Tamaño del subconjunto: 1000

Distribución de etiquetas:
label
FAKE    536
REAL    464
Name: count, dtype: int64


In [57]:
# Aplicamos el preprocesamiento
print("Preprocesando noticias...")
df_sample["text_clean"] = df_sample["text"].apply(preprocesar_texto)
print("¡Listo!")

Preprocesando noticias...
¡Listo!


**Pista 2:** Aplica la vectorización al conjunto de datos para representar las noticias de manera numérica.

In [58]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(df_sample["text_clean"])

In [59]:
print(X_train.toarray())
print("\nFeatures:", len(vectorizer.get_feature_names_out()))

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Features: 19184


In [60]:
pd.DataFrame(X_train.toarray(), columns=[vectorizer.get_feature_names_out()])

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [62]:
y_train = df_sample["label"]
y_train

22216    FAKE
27917    FAKE
25007    FAKE
1377     REAL
32476    FAKE
         ... 
27000    FAKE
25165    FAKE
20813    REAL
11756    REAL
33887    FAKE
Name: label, Length: 1000, dtype: str

**Pista 3**: Entrena el algoritmo `LogisticRegression` de Sklearn. Este algoritmo se basa en aprendizaje supervisado.

In [63]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

### 8. Predicción

¡Muy bien, ya casi has terminado! Ya tenemos nuestro algoritmo entrenado y listo para realizar predicciones, lo único que nos queda es probar qué tal se comporta para noticias que no ha visto nunca.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Utiliza el algoritmo de Machine Learning <code>LogisticRegression</code> para predecir si una nueva noticia que no ha visto nunca es falsa o verdadera.
</div>

**Pista**: Revisa el método `predict()` que tiene la clase `LogisticRegression`. Ten en cuenta que cuando recibas una nueva noticia para la que quieres realizar una predicción debes aplicar sobre ella todas las transformaciones que hemos realizado anteriormente. **Al aplicar la vectorización utiliza únicamente el método `transform()` del objeto `CountVectorizer`**.

##### Lectura de un conjunto de noticias nuevas

In [64]:
# Tomamos las 500 noticias que NO se han utilizado para entrenar el algoritmo
df_test = df_all.iloc[1000:]

print("Preprocesando noticias de test...")
df_test["text_clean"] = df_test["text"].apply(preprocesar_texto)
print("¡Listo!")

Preprocesando noticias de test...
¡Listo!


In [65]:
X_test = df_test["text_clean"]
y_test = df_test["label"]

print(f"Noticias de test: {len(X_test)}")

Noticias de test: 500


##### Preprocesamiento de las noticias con el vectorizador creado anteriormente

In [66]:
# Aplicamos CountVectorizer (solo .transform(), NO .fit())
X_test = vectorizer.transform(X_test)

##### Predicción del tipo de noticia

In [67]:
y_pred = clf.predict(X_test)
y_pred

array(['REAL', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL',
       'REAL', 'REAL', 'REAL', 'FAKE', 'REAL', 'REAL', 'REAL', 'FAKE',
       'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'REAL', 'REAL', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'FAKE',
       'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL',
       'FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'REAL', 'FAKE', 'REAL',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'REAL', 'REAL',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE',
       'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL',
      

In [68]:
print("Predicción:\n", y_pred)
print("\nEtiquetas reales:\n", y_test.values)

Predicción:
 ['REAL' 'REAL' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE'
 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'REAL' 'FAKE'
 'REAL' 'REAL' 'REAL' 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE'
 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'REAL' 'FAKE'
 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE'
 'REAL' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL'
 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL'
 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL'
 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE'
 'FAKE' 'REAL' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'REAL'
 'REAL' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE'
 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'REAL' 'REAL' 

##### Evaluación de los resultados

In [69]:
from sklearn.metrics import accuracy_score

print("Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred)))

Accuracy: 0.950


### 9. Aumentando el conjunto de datos

Nuestro detector de noticias falsas ya funciona bastante bien con solo 1000 noticias. Sin embargo, los algoritmos de Machine Learning suelen funcionar **mejor** cuantos más datos de entrenamiento tienen. Prueba a entrenar el algoritmo con un conjunto de datos de entrenamiento más grande.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Repite el ejercicio anterior utilizando un conjunto de datos de entrenamiento más grande.
</div>

In [70]:
# Leemos 42000 noticias
df_grande = df.sample(n=42000, random_state=42)

print("Preprocesando noticias...")
df_grande["text_clean"] = df_grande["text"].apply(preprocesar_texto)
print("¡Listo!")

Preprocesando noticias...
¡Listo!


In [71]:
# Utilizamos 40000 noticias para entrenar el algoritmo y 2000 para realizar pruebas
X_train, y_train = df_grande["text_clean"][:40000], df_grande["label"][:40000]
X_test, y_test = df_grande["text_clean"][40000:], df_grande["label"][40000:]

print(f"Noticias de entrenamiento: {len(X_train)}")
print(f"Noticias de test: {len(X_test)}")

Noticias de entrenamiento: 40000
Noticias de test: 2000


In [72]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)

In [73]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [74]:
X_test = vectorizer.transform(X_test)

In [75]:
y_pred = clf.predict(X_test)

In [76]:
print("Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred)))

Accuracy: 0.987


### 10. Probando con noticias nuevas

Para terminar, vamos a probar nuestro modelo con noticias que escribamos nosotros mismos. De esta manera podemos ver cómo se comporta el modelo en un escenario más realista.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Escribe manualmente algunas noticias (puedes inventarlas o copiar titulares reales) y utiliza el modelo entrenado para predecir si son falsas o verdaderas. Recuerda que debes aplicar el mismo preprocesamiento y vectorización que aplicaste a los datos de entrenamiento.
</div>

**Pista**: Al recibir una nueva noticia para clasificar, debes:
1. Aplicar la función `preprocesar_texto()` sobre ella.
2. Aplicar **únicamente** el método `.transform()` del `CountVectorizer` (NO `.fit()`).
3. Usar el método `.predict()` del modelo entrenado.
4. Opcionalmente, puedes usar `.predict_proba()` para ver las probabilidades que asigna el modelo a cada clase.

In [77]:
# Definimos algunas noticias de prueba
noticias_nuevas = [
    """DUBAI, Feb 13 (Reuters) - Dubai port giant DP World said on Friday its chairman and chief executive Sultan Ahmed Bin Sulayem had resigned, an announcement that followed mounting pressure over his alleged ties to Jeffrey Epstein.
Bin Sulayem, one of the Middle East's most prominent business figures, is among the highest-profile executives to face scrutiny and be removed from senior roles following the recent release of the Epstein files.""",
    "EXPOSED: Secret documents reveal that the moon landing was filmed in a Hollywood basement by the CIA!!!",
    """WASHINGTON, Feb 12 (Reuters) - An Arizona sheriff is blocking FBI access to key evidence in the investigation into the abduction of U.S. television journalist Savannah Guthrie's mother, impairing its ability to assist in the probe, a U.S. law enforcement official with knowledge of the case told Reuters on Thursday.
The FBI asked Pima County Sheriff Chris Nanos for physical evidence in the case, including a glove and DNA from the home of 84-year-old Nancy Guthrie, to be processed at the FBI's national crime laboratory in Quantico, Virginia, but Nanos has insisted instead on using a private lab in Florida, the official said.""",
    "YOU WONT BELIEVE THIS: Politicians are secretly lizard people controlling the world government!!!"
]

In [78]:
# Preprocesamos las noticias
noticias_procesadas = [preprocesar_texto(n) for n in noticias_nuevas]

# Vectorizamos usando el mismo vectorizer entrenado (solo .transform())
noticias_vect = vectorizer.transform(noticias_procesadas)

# Realizamos las predicciones
predicciones = clf.predict(noticias_vect)
probabilidades = clf.predict_proba(noticias_vect)

# Mostramos los resultados
for noticia, pred, prob in zip(noticias_nuevas, predicciones, probabilidades):
    print(f"Noticia: {noticia[:80]}...")
    print(f"  → Predicción: {pred}")
    print(f"  → Probabilidad FAKE: {prob[0]:.2%} | Probabilidad REAL: {prob[1]:.2%}")
    print()

Noticia: DUBAI, Feb 13 (Reuters) - Dubai port giant DP World said on Friday its chairman ...
  → Predicción: REAL
  → Probabilidad FAKE: 5.27% | Probabilidad REAL: 94.73%

Noticia: EXPOSED: Secret documents reveal that the moon landing was filmed in a Hollywood...
  → Predicción: FAKE
  → Probabilidad FAKE: 99.56% | Probabilidad REAL: 0.44%

Noticia: WASHINGTON, Feb 12 (Reuters) - An Arizona sheriff is blocking FBI access to key ...
  → Predicción: REAL
  → Probabilidad FAKE: 0.09% | Probabilidad REAL: 99.91%

Noticia: YOU WONT BELIEVE THIS: Politicians are secretly lizard people controlling the wo...
  → Predicción: FAKE
  → Probabilidad FAKE: 92.52% | Probabilidad REAL: 7.48%



¡Genial! El modelo es capaz de clasificar noticias que nunca ha visto. Observa cómo el método `.predict_proba()` nos da la **probabilidad** que el modelo asigna a cada clase. Esto es una de las ventajas de la Regresión Logística frente a otros algoritmos de clasificación: no solo nos dice la clase predicha, sino también **cuánto confía** en esa predicción.